# 01 — Carregamento de Dados

**Objetivo:** entender o CSV bruto do ISP-RJ antes de escrever qualquer função.
**Resultado esperado:** ao final, você saberá exatamente quais parâmetros usar no `pd.read_csv()` e o que validar após a leitura.
**Próximo passo:** copiar o código consolidado para `carregar_dados()` em `pipeline.py`.

## Célula 1 — Imports

`pathlib.Path` substitui strings de caminho — funciona igual no Windows, Mac e Linux.

In [8]:
import pandas as pd
from pathlib import Path

## Célula 2 — Caminho do arquivo

`Path.cwd()` retorna a pasta onde o notebook está rodando.
`.parent` sobe um nível (de `notebooks/` para a raiz do projeto).
Assim o caminho funciona independente de onde você abriu o terminal.

In [6]:
RAIZ = Path.cwd().parent
ARQUIVO = RAIZ / "data" / "raw" / "BaseDPEvolucaoMensalCisp.csv"

print("Raiz do projeto:", RAIZ)
print("Arquivo existe?", ARQUIVO.exists())

Raiz do projeto: c:\Users\arthu\OneDrive\Desktop\Projetos\isp-analise
Arquivo existe? True


## Célula 3 — Primeira leitura (sem parâmetros)

Propositalmente simples — queremos ver o que o pandas assume sozinho
e quais problemas aparecem (encoding errado, separador errado).
**Espere ver lixo ou erro aqui — isso é parte do aprendizado.**

In [ ]:
df_teste = pd.read_csv(ARQUIVO)
df_teste.head(2)

## Célula 4 — Leitura correta

Agora com os parâmetros adequados para esse arquivo:

| Parâmetro | Valor | Por quê |
|---|---|---|
| `sep` | `";"` | CSVs do governo BR usam ponto-e-vírgula |
| `encoding` | `"latin-1"` | Padrão de arquivos governamentais com acentos |
| `dtype` | `str` | Carrega tudo como texto; converte depois com controle |
| `low_memory` | `False` | Evita warnings de tipo misto em colunas grandes |

In [10]:
df = pd.read_csv(
    ARQUIVO,
    sep=";",
    encoding="latin-1",
    dtype=str,
    low_memory=False,
)

df.head(3)

,cisp,mes,ano,mes_ano,aisp,risp,munic,mcirc,regiao,hom_doloso,...,cmp,cmba,ameaca,pessoas_desaparecidas,encontro_cadaver,encontro_ossada,pol_militares_mortos_serv,pol_civis_mortos_serv,registro_ocorrencias,fase
0,1,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,0,...,NaN,NaN,21,2,0,0,0,0,578,3
1,4,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,15,6,0,1,0,0,441,3
2,5,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,47,2,1,0,0,0,637,3


## Célula 5 — Dimensões e colunas

`shape` retorna `(linhas, colunas)`.
`columns.tolist()` lista todos os nomes — útil para conferir o que existe
antes de hardcodar nomes no `pipeline.py`.

In [11]:
print("Linhas x Colunas:", df.shape)
print()
print("Colunas disponíveis:")
print(df.columns.tolist())

Linhas x Colunas: (37588, 65)

Colunas disponíveis:
['cisp', 'mes', 'ano', 'mes_ano', 'aisp', 'risp', 'munic', 'mcirc', 'regiao', 'hom_doloso', 'lesao_corp_morte', 'latrocinio', 'cvli', 'hom_por_interv_policial', 'feminicidio', 'letalidade_violenta', 'tentat_hom', 'tentativa_feminicidio', 'lesao_corp_dolosa', 'estupro', 'hom_culposo', 'lesao_corp_culposa', 'roubo_transeunte', 'roubo_celular', 'roubo_em_coletivo', 'roubo_rua', 'roubo_veiculo', 'roubo_carga', 'roubo_comercio', 'roubo_residencia', 'roubo_banco', 'roubo_cx_eletronico', 'roubo_conducao_saque', 'roubo_apos_saque', 'roubo_bicicleta', 'outros_roubos', 'total_roubos', 'furto_veiculos', 'furto_transeunte', 'furto_coletivo', 'furto_celular', 'furto_bicicleta', 'outros_furtos', 'total_furtos', 'sequestro', 'extorsao', 'sequestro_relampago', 'estelionato', 'apreensao_drogas', 'posse_drogas', 'trafico_drogas', 'apreensao_drogas_sem_autor', 'recuperacao_veiculos', 'apf', 'aaapai', 'cmp', 'cmba', 'ameaca', 'pessoas_desaparecidas', 'en

## Célula 6 — Tipos e valores nulos

`dtypes` mostra o tipo de cada coluna — como lemos com `dtype=str`, tudo será `object`.
`isnull().sum()` conta quantos nulos existem por coluna.
Colunas com muitos nulos merecem atenção na etapa de limpeza.

In [12]:
print("Tipos das colunas:")
print(df.dtypes)
print()
print("Nulos por coluna:")
print(df.isnull().sum())

Tipos das colunas:
cisp                         str
mes                          str
ano                          str
mes_ano                      str
aisp                         str
                            ... 
encontro_ossada              str
pol_militares_mortos_serv    str
pol_civis_mortos_serv        str
registro_ocorrencias         str
fase                         str
Length: 65, dtype: object

Nulos por coluna:
cisp                         0
mes                          0
ano                          0
mes_ano                      0
aisp                         0
                            ..
encontro_ossada              0
pol_militares_mortos_serv    0
pol_civis_mortos_serv        0
registro_ocorrencias         0
fase                         0
Length: 65, dtype: int64


## Célula 7 — Inspecionar colunas-chave

Antes de confiar em qualquer análise, vale olhar os valores únicos das colunas de contexto.
Aqui verificamos `mes_ano` (para entender o formato da data) e `regiao` (para ver as categorias).

In [13]:
print("Formato de mes_ano (primeiros valores únicos):")
print(df["mes_ano"].unique()[:10])
print()
print("Valores únicos de regiao:")
print(df["regiao"].unique())

Formato de mes_ano (primeiros valores únicos):
<StringArray>
['2003m01', '2003m02', '2003m03', '2003m04', '2003m05', '2003m06', '2003m07',
 '2003m08', '2003m09', '2003m10']
Length: 10, dtype: str

Valores únicos de regiao:
<StringArray>
[                      'Capital',            'Baixada Fluminense',
                      'Interior', 'Grande NiterÃÂÃÂÃÂÃÂ³i',
                'Grande Niterói']
Length: 5, dtype: str


## Célula 8 — Validar colunas esperadas

Antes de passar o DataFrame adiante, verificamos se as colunas que o pipeline precisa
realmente existem no arquivo. Se alguma faltar, sabemos aqui — não lá na frente.
Essa lógica vai direto para `carregar_dados()` no `pipeline.py`.

In [14]:
COLUNAS_ID = ["cisp", "mes_ano", "aisp", "risp", "munic", "regiao"]
COLUNAS_VIOLENCIA = ["hom_doloso", "latrocinio", "cvli", "letalidade_violenta"]

COLUNAS_NECESSARIAS = COLUNAS_ID + COLUNAS_VIOLENCIA

faltando = [col for col in COLUNAS_NECESSARIAS if col not in df.columns]

if faltando:
    print("ATENCAO — colunas ausentes:", faltando)
else:
    print("Todas as colunas necessárias estão presentes.")

Todas as colunas necessárias estão presentes.


---
## Consolidado — o que vai para `carregar_dados()` no pipeline.py

Quando todas as células acima rodarem sem erro e os resultados fazerem sentido,
copie o bloco abaixo para a função `carregar_dados()` em `pipeline.py`.

```python
def carregar_dados(caminho):
    caminho = Path(caminho)
    df = pd.read_csv(caminho, sep=";", encoding="latin-1", dtype=str, low_memory=False)

    faltando = [col for col in COLUNAS_NECESSARIAS if col not in df.columns]
    if faltando:
        raise ValueError(f"Colunas ausentes no CSV: {faltando}")

    log.info("Dados carregados: %s linhas, %s colunas", *df.shape)
    return df
```

In [15]:
def carregar_dados(caminho: str | Path) -> pd.DataFrame:
    
    caminho = Path(caminho)

    try:
        df = pd.read_csv(
            caminho,
            sep=";",
            encoding="latin-1",
            dtype=str,
            low_memory=False,
        )

    except FileNotFoundError:
        log.error("Arquivo não encontrado: %s", caminho)
        raise

    except pd.errors.EmptyDataError:
        log.error("O arquivo está vazio: %s", caminho)
        raise ValueError(f"Arquivo vazio: {caminho}")

    faltando = [col for col in COLUNAS_ID + COLUNAS_VIOLENCIA if col not in df.columns]
    if faltando:
        raise ValueError(f"Colunas ausentes no CSV: {faltando}")

    log.info("Dados carregados: %s linhas, %s colunas", *df.shape)
    return df

In [18]:
import sys                                                                                                                                                                               
sys.path.append(str(RAIZ))  # RAIZ já foi definida na Célula 2
                                                                                                                                                                                           
from pipeline import carregar_dados                                                                                                                                                    

df = carregar_dados(ARQUIVO)
print(df.shape)
df.head(5)

2026-05-06 20:13:34 [INFO] Dados carregados: 37588 linhas, 65 colunas


(37588, 65)


,cisp,mes,ano,mes_ano,aisp,risp,munic,mcirc,regiao,hom_doloso,...,cmp,cmba,ameaca,pessoas_desaparecidas,encontro_cadaver,encontro_ossada,pol_militares_mortos_serv,pol_civis_mortos_serv,registro_ocorrencias,fase
0,1,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,0,...,NaN,NaN,21,2,0,0,0,0,578,3
1,4,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,15,6,0,1,0,0,441,3
2,5,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,47,2,1,0,0,0,637,3
3,6,1,2003,2003m01,1,1,Rio de Janeiro,3304557,Capital,6,...,NaN,NaN,26,2,1,0,0,0,473,3
4,7,1,2003,2003m01,1,1,Rio de Janeiro,3304557,Capital,4,...,NaN,NaN,10,1,3,0,0,0,147,3
